[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fvalenzuelag/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/main/RA1/IL1.1/4-langchain_memory.ipynb)


# 4. LangChain Memory - Gestión de Contexto Conversacional

## Objetivos de Aprendizaje
- Comprender la importancia de la memoria en conversaciones con LLMs
- Implementar diferentes tipos de memoria con LangChain
- Gestionar el contexto de conversaciones largas
- Optimizar el uso de tokens con estrategias de memoria

## ¿Por qué es Importante la Memoria?

Los LLMs son **stateless** por naturaleza: no recuerdan conversaciones anteriores. La memoria permite:
- **Contexto conversacional**: Referirse a mensajes anteriores
- **Personalización**: Recordar preferencias del usuario
- **Continuidad**: Mantener hilos de conversación coherentes
- **Experiencia natural**: Conversaciones que se sienten humanas

## Tipos de Memoria en LangChain

1. **ConversationBufferMemory**: Mantiene todo el historial
2. **ConversationSummaryMemory**: Resume conversaciones largas
3. **ConversationBufferWindowMemory**: Mantiene solo los N mensajes más recientes
4. **ConversationSummaryBufferMemory**: Combina resumen + buffer reciente

In [1]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q langchain-openai python-dotenv


In [2]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "GOOGLE_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
    os.environ.setdefault("LLM_MODEL", "llama-3.3-70b-versatile")
    os.environ.setdefault("LLM_MODEL_SMALL", "llama-3.1-8b-instant")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

# Importar bibliotecas necesarias para memoria
from langchain_openai import ChatOpenAI
# NOTA: las clases clásicas de memoria (ConversationBufferMemory,
# ConversationSummaryMemory, ConversationBufferWindowMemory) que se describen
# más abajo están DEPRECADAS en LangChain v1. Este notebook usa el reemplazo
# oficial: RunnableWithMessageHistory. Si quieres experimentar con las clásicas:
#     from langchain_classic.memory import ConversationBufferMemory
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

import os

print("✓ Bibliotecas de memoria importadas correctamente")

✓ Bibliotecas de memoria importadas correctamente


In [3]:
# Configuración del modelo para memoria
try:
    llm = ChatOpenAI(
        base_url=os.getenv("LLM_BASE_URL"),
        api_key=os.getenv("LLM_API_KEY"),
        model=os.getenv("LLM_MODEL_SMALL", "llama-3.1-8b-instant"),
        temperature=0.1
    )
    
    print("✓ Modelo configurado para experimentos de memoria")
    print(f"Modelo: {llm.model_name}")
    
except Exception as e:
    print(f"✗ Error en configuración: {e}")
    print("Verifica las variables de entorno")

✓ Modelo configurado para experimentos de memoria
Modelo: llama-3.1-8b-instant


In [4]:
# Prompt con historial + entrada del usuario
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Cadena = prompt + modelo
chain = prompt | llm

# Almacén de historiales
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Envolver con memoria
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

<ruta-local>:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 1. ConversationBufferMemory - Memoria Completa

Esta memoria mantiene **todo** el historial de la conversación. Es la más simple pero puede consumir muchos tokens.

In [5]:
# Ejemplo básico con RunnableWithMessageHistory

# Prompt con historial + entrada
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Cadena = prompt + modelo
chain = prompt | llm

# Almacén de memorias por sesión
store = {}

def get_session_history(session_id: str):
    """Devuelve (o crea) el historial completo para la sesión."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Envolver con RunnableWithMessageHistory
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def ejemplo_buffer_memory():
    print("=== CONVERSATIONBUFFERMEMORY ===")
    print("Mantiene todo el historial de conversación\n")
    
    session_id = "demo_session"

    try:
        # Primera interacción
        print("1. Primera pregunta:")
        response1 = conversation.invoke(
            {"input": "Mi nombre es Ana y soy programadora Python"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response1.content}\n")

        # Segunda interacción
        print("2. Segunda pregunta:")
        response2 = conversation.invoke(
            {"input": "¿Cuál es mi nombre y profesión?"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response2.content}\n")

        # Tercera interacción
        print("3. Tercera pregunta:")
        response3 = conversation.invoke(
            {"input": "¿Qué lenguaje de programación mencioné?"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response3.content}\n")

        # Mostrar historial
        print("=== CONTENIDO DE LA MEMORIA ===")
        history = store[session_id].messages
        for i, msg in enumerate(history, 1):
            print(f"{i}. {msg.type}: {msg.content}")

    except Exception as e:
        print(f"Error: {e}")

# Ejecutar
ejemplo_buffer_memory()


=== CONVERSATIONBUFFERMEMORY ===
Mantiene todo el historial de conversación

1. Primera pregunta:


<ruta-local>:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Respuesta: Hola Ana, soy un asistente útil y estoy aquí para ayudarte con cualquier pregunta o problema que tengas relacionado con la programación en Python. ¿En qué puedo ayudarte hoy? ¿Estás trabajando en un proyecto específico o tienes alguna duda sobre un tema en particular?

2. Segunda pregunta:


Respuesta: Tu nombre es Ana y eres programadora Python.

3. Tercera pregunta:


Respuesta: Mencionaste Python como lenguaje de programación.

=== CONTENIDO DE LA MEMORIA ===
1. human: Mi nombre es Ana y soy programadora Python
2. ai: Hola Ana, soy un asistente útil y estoy aquí para ayudarte con cualquier pregunta o problema que tengas relacionado con la programación en Python. ¿En qué puedo ayudarte hoy? ¿Estás trabajando en un proyecto específico o tienes alguna duda sobre un tema en particular?
3. human: ¿Cuál es mi nombre y profesión?
4. ai: Tu nombre es Ana y eres programadora Python.
5. human: ¿Qué lenguaje de programación mencioné?
6. ai: Mencionaste Python como lenguaje de programación.


## 2. ConversationBufferWindowMemory - Ventana Deslizante

Esta memoria mantiene solo los **N mensajes más recientes**, útil para controlar el uso de tokens.

In [6]:
# Prompt con historial + entrada
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Cadena = prompt + modelo
chain = prompt | llm

# Almacén de memorias por sesión
store = {}

class WindowChatMessageHistory(BaseChatMessageHistory):
    """Historial de chat que mantiene solo los últimos k intercambios."""
    
    def __init__(self, k: int = 2):
        self.k = k
        self._messages = []
    
    @property
    def messages(self):
        # Mantener solo los últimos k intercambios (k*2 mensajes: user + assistant)
        return self._messages[-(self.k * 2):]
    
    def add_message(self, message):
        self._messages.append(message)
    
    def clear(self):
        self._messages.clear()

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """Devuelve el historial de ventana para la sesión."""
    if session_id not in store:
        store[session_id] = WindowChatMessageHistory(k=2)
    return store[session_id]

# Envolver con RunnableWithMessageHistory
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# Ejemplo
def ejemplo_window_memory():
    print("=== CONVERSATION BUFFER WINDOW MEMORY (k=2) ===")
    print("Mantiene solo los 2 intercambios más recientes\n")
    
    session_id = "demo_window"
    inputs = [
        "Mi nombre es Carlos y tengo 30 años",
        "Trabajo como diseñador gráfico", 
        "Me gusta el café y la música jazz",
        "¿Puedes recordar mi edad?",
        "¿Cuál es mi profesión?"
    ]
    
    try:
        for i, user_input in enumerate(inputs, 1):
            print(f"{'='*20} INTERACCIÓN {i} {'='*20}")
            print(f"👤 Usuario: {user_input}")
            
            response = conversation.invoke(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            )
            print(f"🤖 Asistente: {response.content}\n")
            
            # Obtener el historial
            history = get_session_history(session_id)
            
            # Mostrar comparación clara
            total_messages = len(history._messages)
            visible_messages = len(history.messages)
            
            print(f"📊 ESTADO DE LA MEMORIA:")
            print(f"   💾 Total almacenado: {total_messages} mensajes")
            print(f"   👁️  Visible al modelo: {visible_messages} mensajes")
            print(f"   🗑️  Mensajes descartados: {total_messages - visible_messages}")
            
            # Mensajes almacenados totalmente
            print(f"\n📚 HISTORIAL COMPLETO ALMACENADO ({total_messages} mensajes):")
            if total_messages == 0:
                print("     (Ningún mensaje aún)")
            else:
                for j, msg in enumerate(history._messages, 1):
                    role = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                    content = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
                    # Marcar si está en la ventana visible
                    is_visible = j > total_messages - visible_messages
                    marker = "✅" if is_visible else "❌"
                    print(f"     {j}. {marker} {role}: {content}")
            
            # Lo que ve el modelo
            print(f"\n🔍 VENTANA VISIBLE AL MODELO ({visible_messages} mensajes):")
            if visible_messages == 0:
                print("     (Ningún mensaje visible)")
            else:
                for j, msg in enumerate(history.messages, 1):
                    role = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                    content = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
                    print(f"     {j}. ✅ {role}: {content}")
            
            print("\n" + "="*60 + "\n")
            
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar
ejemplo_window_memory()

=== CONVERSATION BUFFER WINDOW MEMORY (k=2) ===
Mantiene solo los 2 intercambios más recientes

==================== INTERACCIÓN 1 ====================
👤 Usuario: Mi nombre es Carlos y tengo 30 años


<ruta-local>:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy? ¿Tienes algún tema en particular que te gustaría discutir o necesitas información sobre algo?

📊 ESTADO DE LA MEMORIA:
   💾 Total almacenado: 2 mensajes
   👁️  Visible al modelo: 2 mensajes
   🗑️  Mensajes descartados: 0

📚 HISTORIAL COMPLETO ALMACENADO (2 mensajes):
     1. ✅ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ✅ 🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy...

🔍 VENTANA VISIBLE AL MODELO (2 mensajes):
     1. ✅ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ✅ 🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy...


==================== INTERACCIÓN 2 ====================
👤 Usuario: Trabajo como diseñador gráfico


🤖 Asistente: Eso es genial, Carlos. El diseño gráfico es un campo emocionante y creativo. ¿Qué tipo de proyectos te gustan trabajar más? ¿Eres especializado en logotipos, publicidad, ilustraciones o algo más?

¿Tienes algún proyecto en particular que estés trabajando en este momento o algún desafío que te gustaría superar en tu carrera como diseñador gráfico?

📊 ESTADO DE LA MEMORIA:
   💾 Total almacenado: 4 mensajes
   👁️  Visible al modelo: 4 mensajes
   🗑️  Mensajes descartados: 0

📚 HISTORIAL COMPLETO ALMACENADO (4 mensajes):
     1. ✅ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ✅ 🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy...
     3. ✅ 👤 Usuario: Trabajo como diseñador gráfico
     4. ✅ 🤖 Asistente: Eso es genial, Carlos. El diseño gráfico es un campo emocion...

🔍 VENTANA VISIBLE AL MODELO (4 mensajes):
     1. ✅ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ✅ 🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte h

🤖 Asistente: Tienes buen gusto, Carlos. El café y la música jazz son dos de las cosas más relajantes y creativas que existen. Me imagino que debes disfrutar de un buen café mientras trabajas en tus proyectos de diseño gráfico, y que la música jazz te inspire a crear algo nuevo y emocionante.

¿Te gustaría compartir conmigo algún artista de jazz que te inspire o algún café que te encante visitar?

📊 ESTADO DE LA MEMORIA:
   💾 Total almacenado: 6 mensajes
   👁️  Visible al modelo: 4 mensajes
   🗑️  Mensajes descartados: 2

📚 HISTORIAL COMPLETO ALMACENADO (6 mensajes):
     1. ❌ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ❌ 🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy...
     3. ✅ 👤 Usuario: Trabajo como diseñador gráfico
     4. ✅ 🤖 Asistente: Eso es genial, Carlos. El diseño gráfico es un campo emocion...
     5. ✅ 👤 Usuario: Me gusta el café y la música jazz
     6. ✅ 🤖 Asistente: Tienes buen gusto, Carlos. El café y la música jazz son dos ...

🔍

🤖 Asistente: Lo siento, pero no tengo la capacidad de recordar información personal como la edad de los usuarios. Nuestro conocimiento se basa en la información que se me proporciona en el momento de la conversación, y no tengo acceso a información previa sobre ti.

Si deseas compartir tu edad conmigo, estaré encantado de saberlo y podemos hablar sobre temas relacionados con tu edad y experiencia.

📊 ESTADO DE LA MEMORIA:
   💾 Total almacenado: 8 mensajes
   👁️  Visible al modelo: 4 mensajes
   🗑️  Mensajes descartados: 4

📚 HISTORIAL COMPLETO ALMACENADO (8 mensajes):
     1. ❌ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ❌ 🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy...
     3. ❌ 👤 Usuario: Trabajo como diseñador gráfico
     4. ❌ 🤖 Asistente: Eso es genial, Carlos. El diseño gráfico es un campo emocion...
     5. ✅ 👤 Usuario: Me gusta el café y la música jazz
     6. ✅ 🤖 Asistente: Tienes buen gusto, Carlos. El café y la música jazz son dos ...


🤖 Asistente: No tengo información sobre tu profesión. Nuestra conversación comenzó cuando mencionaste que te gustaba el café y la música jazz, pero no tengo conocimiento sobre tu trabajo o profesión. Si deseas hablar sobre tu carrera o intereses laborales, estaré encantado de escucharte y ofrecerte consejos o recomendaciones.

📊 ESTADO DE LA MEMORIA:
   💾 Total almacenado: 10 mensajes
   👁️  Visible al modelo: 4 mensajes
   🗑️  Mensajes descartados: 6

📚 HISTORIAL COMPLETO ALMACENADO (10 mensajes):
     1. ❌ 👤 Usuario: Mi nombre es Carlos y tengo 30 años
     2. ❌ 🤖 Asistente: Hola Carlos, un placer conocerte. ¿En qué puedo ayudarte hoy...
     3. ❌ 👤 Usuario: Trabajo como diseñador gráfico
     4. ❌ 🤖 Asistente: Eso es genial, Carlos. El diseño gráfico es un campo emocion...
     5. ❌ 👤 Usuario: Me gusta el café y la música jazz
     6. ❌ 🤖 Asistente: Tienes buen gusto, Carlos. El café y la música jazz son dos ...
     7. ✅ 👤 Usuario: ¿Puedes recordar mi edad?
     8. ✅ 🤖 Asistente: L

## 3. ConversationSummaryMemory - Resumen Inteligente

Esta memoria **resume** conversaciones largas en lugar de mantener todo el texto completo, ahorrando tokens significativamente.

In [7]:

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Función para resumir automáticamente cuando hay muchos mensajes
def auto_summarize(session_id: str, max_messages=6):
    history = get_session_history(session_id)
    
    if len(history.messages) > max_messages:
        # Mensajes a resumir (todos excepto los últimos 2)
        messages_to_summarize = history.messages[:-2]
        
        # Crear texto para resumir
        conversation_text = ""
        for msg in messages_to_summarize:
            role = "Usuario" if msg.type == "human" else "Asistente"
            conversation_text += f"{role}: {msg.content}\n"
        
        # Generar resumen
        summary_response = llm.invoke(f"Resume esta conversación en 2-3 líneas:\n{conversation_text}")
        summary = summary_response.content
        
        # Reemplazar mensajes antiguos con el resumen
        recent_messages = history.messages[-2:]
        history.clear()
        history.add_ai_message(f"[RESUMEN]: {summary}")
        history.messages.extend(recent_messages)

# Crear conversación
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

conversation = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def ejemplo_summary_memory():
    print("=== CONVERSATION SUMMARY MEMORY ===")
    print("Resume conversaciones largas para ahorrar tokens\n")
    
    session_id = "summary_session"
    
    # Conversación de ejemplo
    inputs = [
        "Hola, soy María González, ingeniera de software de 35 años",
        "Trabajo en una startup de fintech en Madrid desarrollando pagos digitales",
        "Usamos React, Node.js, Docker y Kubernetes en nuestros proyectos",
        "Mi mayor desafío es la latencia en transacciones internacionales",
        "También trabajo en mejorar la UX de nuestra app móvil",
        "¿Puedes resumir quién soy y cuáles son mis principales desafíos?"
    ]
    
    try:
        for i, user_input in enumerate(inputs, 1):
            print(f"{'='*15} INTERACCIÓN {i} {'='*15}")
            print(f"👤 Usuario: {user_input}")
            
            # Resumir automáticamente si es necesario
            auto_summarize(session_id)
            
            response = conversation.invoke(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            )
            print(f"🤖 Asistente: {response.content}\n")
            
            # Mostrar estado de la memoria
            history = get_session_history(session_id)
            total_messages = len(history.messages)
            
            print(f"📊 ESTADO DE LA MEMORIA:")
            print(f"   💾 Total mensajes: {total_messages}")
            
            # Verificar si hay resumen
            has_summary = any("[RESUMEN]" in msg.content for msg in history.messages if hasattr(msg, 'content'))
            print(f"   📝 Tiene resumen: {'✅ Sí' if has_summary else '❌ No'}")
            
            print(f"\n💬 CONTENIDO ACTUAL DE LA MEMORIA:")
            for j, msg in enumerate(history.messages, 1):
                role = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                content = msg.content
                
                # Destacar si es un resumen
                if "[RESUMEN]" in content:
                    role = "📝 Resumen"
                    content = content.replace("[RESUMEN]: ", "")
                
                # Truncar si es muy largo
                if len(content) > 80:
                    content = content[:80] + "..."
                
                print(f"   {j}. {role}: {content}")
            
            print("\n" + "="*50 + "\n")
            
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar
ejemplo_summary_memory()

=== CONVERSATION SUMMARY MEMORY ===
Resume conversaciones largas para ahorrar tokens

=============== INTERACCIÓN 1 ===============
👤 Usuario: Hola, soy María González, ingeniera de software de 35 años


<ruta-local>:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


🤖 Asistente: Hola María, un placer conocerte. ¿En qué puedo ayudarte hoy? ¿Tienes algún proyecto de software en el que necesites asistencia o simplemente quieres charlar sobre tecnología?

📊 ESTADO DE LA MEMORIA:
   💾 Total mensajes: 2
   📝 Tiene resumen: ❌ No

💬 CONTENIDO ACTUAL DE LA MEMORIA:
   1. 👤 Usuario: Hola, soy María González, ingeniera de software de 35 años
   2. 🤖 Asistente: Hola María, un placer conocerte. ¿En qué puedo ayudarte hoy? ¿Tienes algún proye...


=============== INTERACCIÓN 2 ===============
👤 Usuario: Trabajo en una startup de fintech en Madrid desarrollando pagos digitales


🤖 Asistente: Eso suena emocionante. La fintech es un sector en constante evolución y los pagos digitales están cambiando la forma en que las personas y las empresas realizan transacciones financieras.

¿Cuál es tu rol específico en la startup? ¿Eres desarrolladora de backend, frontend o trabajas en la integración de sistemas? ¿Qué tecnologías y herramientas estás utilizando en tu trabajo diario?

📊 ESTADO DE LA MEMORIA:
   💾 Total mensajes: 4
   📝 Tiene resumen: ❌ No

💬 CONTENIDO ACTUAL DE LA MEMORIA:
   1. 👤 Usuario: Hola, soy María González, ingeniera de software de 35 años
   2. 🤖 Asistente: Hola María, un placer conocerte. ¿En qué puedo ayudarte hoy? ¿Tienes algún proye...
   3. 👤 Usuario: Trabajo en una startup de fintech en Madrid desarrollando pagos digitales
   4. 🤖 Asistente: Eso suena emocionante. La fintech es un sector en constante evolución y los pago...


=============== INTERACCIÓN 3 ===============
👤 Usuario: Usamos React, Node.js, Docker y Kubernetes en nuestros proyec

🤖 Asistente: Una combinación interesante de tecnologías. React es una excelente opción para el frontend, ya que permite crear interfaces de usuario interactivas y escalables. Node.js es un buen elección para el backend, ya que ofrece una plataforma de ejecución de JavaScript en el lado del servidor.

Docker y Kubernetes son herramientas muy útiles para la gestión de contenedores y la orquestación de aplicaciones en entornos de producción. ¿Cómo estás utilizando Docker y Kubernetes en tu proyecto? ¿Tienes experiencia con la creación de imágenes Docker y la configuración de deployments en Kubernetes?

¿Qué tipo de desafíos estás enfrentando en tu trabajo actual? ¿Hay algún proyecto o tecnología que te gustaría aprender más sobre en el futuro?

📊 ESTADO DE LA MEMORIA:
   💾 Total mensajes: 6
   📝 Tiene resumen: ❌ No

💬 CONTENIDO ACTUAL DE LA MEMORIA:
   1. 👤 Usuario: Hola, soy María González, ingeniera de software de 35 años
   2. 🤖 Asistente: Hola María, un placer conocerte. ¿En qué puedo

🤖 Asistente: La latencia en transacciones internacionales es un desafío común en la industria de los pagos digitales. La latencia se refiere al tiempo que tarda en procesar una transacción, y en el caso de transacciones internacionales, esto puede ser especialmente crítico debido a las diferencias en los horarios de negocio y las redes de comunicación.

Hay varias razones por las que la latencia puede ser un problema en transacciones internacionales, incluyendo:

* Diferencias en los horarios de negocio: Los bancos y las instituciones financieras pueden tener horarios de negocio diferentes, lo que puede causar retrasos en la procesamiento de transacciones.
* Redes de comunicación lentas: Las redes de comunicación pueden ser lentas o inestables, lo que puede causar retrasos en la transmisión de datos.
* Regulaciones y normas: Las regulaciones y normas financieras pueden variar de un país a otro, lo que puede causar retrasos en la procesamiento de transacciones.

¿Qué estrategias estás c

🤖 Asistente: Me alegra saber que estás trabajando en mejorar la experiencia del usuario (UX) de tu app móvil. La UX es un aspecto crucial para cualquier aplicación, ya que puede influir en la satisfacción del usuario y la retención de la aplicación.

¿Cuáles son algunos de los desafíos que estás enfrentando al mejorar la UX de tu app móvil? ¿Has realizado alguna investigación de usuarios para entender mejor sus necesidades y preferencias?

Algunas preguntas que podrían ayudarme a entender mejor tu situación son:

* ¿Cuál es el objetivo principal de la app móvil? ¿Es para realizar pagos, gestionar cuentas, o algo más?
* ¿Cuál es el público objetivo de la app móvil? ¿Son usuarios jóvenes o adultos, y qué características tienen en común?
* ¿Qué características de la app móvil están causando problemas para los usuarios? ¿Es la navegación, la interfaz de usuario, o algo más?
* ¿Has realizado alguna prueba de usabilidad para identificar áreas de mejora?

Algunas estrategias que podrías consi

🤖 Asistente: Claro, te resumiré quién eres y cuáles son tus principales desafíos:

**Quién eres:** Eres María González, una ingeniera de software de 35 años que trabaja en una startup de fintech en Madrid. Desarrollas pagos digitales con tecnologías como React, Node.js, Docker y Kubernetes.

**Principales desafíos:**

1. **Latencia en transacciones internacionales**: Estás enfrentando desafíos para reducir la latencia en transacciones internacionales, lo que puede afectar la experiencia del usuario y la eficiencia de las transacciones.
2. **Mejora de la UX de la app móvil**: Estás trabajando en mejorar la experiencia del usuario (UX) de la app móvil, lo que incluye identificar áreas de mejora, desarrollar una interfaz de usuario clara y fácil de usar, y ofrecer características de personalización.

Espero que esta información sea útil. ¿Hay algo más en lo que pueda ayudarte?

📊 ESTADO DE LA MEMORIA:
   💾 Total mensajes: 7
   📝 Tiene resumen: ✅ Sí

💬 CONTENIDO ACTUAL DE LA MEMORIA:
   1.

## Consideraciones Técnicas y Mejores Prácticas

### Selección del Tipo de Memoria

| Tipo | Cuándo Usarlo | Ventajas | Desventajas |
|------|---------------|----------|-------------|
| **Buffer** | Conversaciones cortas | Contexto completo | Alto consumo de tokens |
| **Window** | Contexto reciente importante | Eficiente en tokens | Puede perder información clave |
| **Summary** | Conversaciones muy largas | Balance eficiencia/contexto | Pérdida de detalles específicos |

### Mejores Prácticas:

1. **Gestión de Tokens**:
   - Monitorea el uso de tokens regularmente
   - Establece límites máximos para evitar costos excesivos
   - Considera el costo vs. calidad del contexto

2. **Selección Estratégica**:
   - Usa Buffer para sesiones cortas e importantes
   - Usa Window para conversaciones con contexto limitado
   - Usa Summary para sesiones largas de asistencia

3. **Optimización**:
   - Limpia memoria periódicamente si es necesario
   - Implementa estrategias híbridas según el caso de uso
   - Considera almacenamiento persistente para memoria a largo plazo

## Ejercicios Prácticos

### Ejercicio 1: Análisis de Consumo
Implementa un sistema que monitoree y reporte el uso de tokens con diferentes tipos de memoria.

### Ejercicio 2: Memoria Híbrida
Diseña una estrategia que combine multiple tipos de memoria según el contexto.

### Ejercicio 3: Persistencia
Extiende el chatbot para guardar y cargar memoria entre sesiones.

## Conceptos Clave Aprendidos

1. **Importancia de la memoria** en conversaciones naturales
2. **Tipos de memoria** y sus casos de uso específicos
3. **Balance** entre contexto y eficiencia de tokens
4. **Implementación práctica** con LangChain
5. **Estrategias de optimización** para diferentes escenarios

## Conclusión del Módulo IL1.1

Has completado la introducción a LLMs y conexiones API. Los conceptos aprendidos:

1. **APIs directas** vs **frameworks** como LangChain
2. **Streaming** para mejor experiencia de usuario
3. **Memoria** para conversaciones contextuales
4. **Mejores prácticas** de seguridad y optimización

### Próximos Pasos
En **IL1.2** exploraremos técnicas avanzadas de **prompt engineering** incluyendo zero-shot, few-shot, y chain-of-thought prompting.